In [0]:
CREATE OR REPLACE TABLE cfpb_risk.gold.issue_clusters_daily AS
WITH daily_base AS (
  SELECT
    institution_display_name, 
    rssd_id, 
    date_received, 
    product, 
    issue, 
    COUNT(*) AS complaint_count
  FROM cfpb_risk.silver.cfpb_complaints_clean
  GROUP BY 
    institution_display_name,
    rssd_id, 
    date_received, 
    product, 
    issue    
),
daily_enriched AS (
  SELECT
    institution_display_name, 
    rssd_id, 
    date_received, 
    product, 
    issue, 
    complaint_count,
    LAG(complaint_count, 1)
      OVER (
        PARTITION BY 
          rssd_id,
          product, 
          issue
        ORDER BY date_received
      ) AS prior_day_complaint_count,
    AVG(complaint_count) OVER (
      PARTITION BY 
        rssd_id,
        product, 
        issue
      ORDER BY date_received
      ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS rolling_3_day_avg,
    SUM(complaint_count) OVER (
      PARTITION BY 
        rssd_id,
        date_received)
      AS bank_total_complaints_daily
  FROM daily_base
)
SELECT
  institution_display_name,
  rssd_id,
  date_received,
  product,
  issue,
  complaint_count,
  prior_day_complaint_count,
  rolling_3_day_avg,
  bank_total_complaints_daily,
  complaint_count/NULLIF(bank_total_complaints_daily,0) as share_of_bank_complaints
FROM daily_enriched